## Latar Belakang Kasus
TidarMart adalah platform e-commerce regional yang melayani delapan kota di Jawa Tengah dan DIY. Dalam
enam bulan terakhir, jumlah transaksi tumbuh pesat hingga lebih dari dua juta baris log pesanan. Data tersebut
tersebar di banyak sumber dengan format berbeda: log transaksi berupa CSV yang dipecah per bagian, profil
pelanggan dan katalog produk berupa JSON Lines dengan struktur yang tidak seragam, serta aliran transaksi
baru yang masuk terus-menerus sepanjang hari.
Tim bisnis mengeluhkan tiga hal: (1) laporan manual di spreadsheet sudah tidak sanggup membuka datanya, (2)
angka laporan sering tidak konsisten karena data kotor dan duplikat, serta (3) tidak ada sistem yang dapat
memberi peringatan dini ketika pesanan berisiko dibatalkan. Anda ditunjuk sebagai Data Engineer sekaligus
Data Scientist tunggal yang harus membangun prototipe platform analitik end-to-end: dari penyimpanan
terdistribusi, pembersihan data, analitik batch, penyimpanan NoSQL, pemrosesan streaming, hingga model
prediktif yang disimpan dan dipakai ulang.

### Tahap 1
Membuat Struktur direktori di HDFS, Mengunggah Data (bukti ukuran, jumlah blok, dan faktor replikasi), dan Menjelaskan mengapa data mentah dan data bersih dipisah direktori, dan apa arti replikasi = 1 pada VM pseudo-distributed dibanding klaster nyata.

In [8]:
# !hdfs version
!hdfs dfs -mkdir -p /tidarmart/raw/transaksi
!hdfs dfs -mkdir -p /tidarmart/raw/stream_input
!hdfs dfs -mkdir -p /tidarmart/raw/pelanggan
!hdfs dfs -mkdir -p /tidarmart/raw/produk
!hdfs dfs -mkdir -p /tidarmart/raw/models

!hdfs dfs -put -f data_proyek/transaksi/* /tidarmart/raw/transaksi
!hdfs dfs -put -f data_proyek/stream_input/* /tidarmart/raw/stream_input
!hdfs dfs -put -f data_proyek/pelanggan.json /tidarmart/raw/pelanggan
!hdfs dfs -put -f data_proyek/produk.json /tidarmart/raw/produk

!hdfs dfs -ls /tidarmart/raw
print()
!hdfs dfs -count /tidarmart/raw/*
print()
print("Data Berhasil Diunggah!")

Found 5 items
drwxr-xr-x   - asta supergroup          0 2026-11-26 03:34 /tidarmart/raw/models
drwxr-xr-x   - asta supergroup          0 2026-11-26 04:02 /tidarmart/raw/pelanggan
drwxr-xr-x   - asta supergroup          0 2026-11-26 04:02 /tidarmart/raw/produk
drwxr-xr-x   - asta supergroup          0 2026-11-26 04:02 /tidarmart/raw/stream_input
drwxr-xr-x   - asta supergroup          0 2026-11-26 04:02 /tidarmart/raw/transaksi

           1            0                  0 /tidarmart/raw/models
           1            1            6307668 /tidarmart/raw/pelanggan
           1            1             697983 /tidarmart/raw/produk
           1           20            9817421 /tidarmart/raw/stream_input
           1            4          196349436 /tidarmart/raw/transaksi

Data Berhasil Diunggah!


In [9]:
!hdfs dfs -du -h /tidarmart/raw
!hdfs fsck /tidarmart/raw -files -blocks

0        0        /tidarmart/raw/models
6.0 M    6.0 M    /tidarmart/raw/pelanggan
681.6 K  681.6 K  /tidarmart/raw/produk
9.4 M    9.4 M    /tidarmart/raw/stream_input
187.3 M  187.3 M  /tidarmart/raw/transaksi
Connecting to namenode via http://localhost:9870/fsck?ugi=asta&files=1&blocks=1&path=%2Ftidarmart%2Fraw
FSCK started by asta (auth:SIMPLE) from /127.0.0.1 for path /tidarmart/raw at Thu Nov 26 04:02:21 UTC 2026

/tidarmart/raw <dir>
/tidarmart/raw/models <dir>
/tidarmart/raw/pelanggan <dir>
/tidarmart/raw/pelanggan/pelanggan.json 6307668 bytes, replicated: replication=1, 1 block(s):  OK
0. BP-1650105502-10.0.2.15-1788694273454:blk_1073741927_1103 len=6307668 Live_repl=1

/tidarmart/raw/produk <dir>
/tidarmart/raw/produk/produk.json 697983 bytes, replicated: replication=1, 1 block(s):  OK
0. BP-1650105502-10.0.2.15-1788694273454:blk_1073741928_1104 len=697983 Live_repl=1

/tidarmart/raw/stream_input <dir>
/tidarmart/raw/stream_input/stream_01.csv 491161 bytes, replicated: replic

#### mengapa data mentah dan data bersih dipisah direktori, dan apa arti replikasi = 1 pada VM pseudo-distributed dibanding klaster nyata.
Data mentah dan data bersih kita pisah direktori untuk menjaga keaslian data, ini sangat penting karena untuk membuktikan bahwa yang dipakai itu benar benar data dari tidarmart yang belum diolah, kita bisa membuktikannya melalui data mentah.

Pemisahan ini juga mempermudah kita jika setelah dilakukannya pembersihan data terdapat kesalahan dalam pembersihannya, jika kita menggabungkannya maka kita telah kehilangan data mentah tadi (karena tertimpa) dan pembersihan tidak berjalan dengan baik(karena tidak bisa diulang). Jika kita menggunakan data yang terpisah dan terjadi kesalahan kita dapat membersihkannya ulang karena data mentahnya masih utuh pada direktori yang berbeda.

